In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Geometry-V1 D2 independent confirmation

Run once from top to bottom. This is a bounded handoff only; `science_denominator=0`.


In [ ]:
import json, os, pathlib, subprocess, sys, time
from google.colab import userdata
D2_RUNNER_EXACT='4b276ca031a97988fac39a4430c9eaf8dbc0d0f2'
SOURCE_ROOT=pathlib.Path('/content/drive/MyDrive/CEG-WM/Geometry-V1/DIRECTION_ALL_LAYER/Geometry-V1-Direction-All-Layer-41742d462d62-20260827T123709Z')
OUTPUT_ROOT=pathlib.Path('/content/drive/MyDrive/CEG-WM/Geometry-V1/D2')
RUNNER_PATH='experiments/run_geometry_v1_qk_d2_independent_confirmation_operational.py'
SUCCESS_PREFIX='CEGWM_GEOMETRY_V1_QK_D2 '; FAILURE_PREFIX='CEGWM_GEOMETRY_V1_QK_D2_FAILURE '; MAX_CONTROL_BYTES=1024
FAILED=False; ATTEMPTED=False; hf_token=None; runner_env={}
def fail(stage):
 global FAILED
 if not FAILED: FAILED=True; print('CEGWM_GEOMETRY_V1_QK_D2_HANDOFF_FAILURE '+json.dumps({'stage':stage,'error_class':'handoff_error'},sort_keys=True,separators=(',',':')))


In [ ]:
if not FAILED:
 try:
  if not SOURCE_ROOT.is_dir(): raise RuntimeError()
  repo=pathlib.Path('/content/Geometry-V1-D2')
  if repo.exists(): raise RuntimeError()
  subprocess.run(['git','clone','--no-checkout','https://github.com/RICHAAARC/CEG-WM.git',str(repo)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
  subprocess.run(['git','checkout','--detach',D2_RUNNER_EXACT],cwd=repo,check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
  execution_commit=subprocess.run(['git','rev-parse','HEAD'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip(); checkout_clean=not subprocess.run(['git','status','--porcelain'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip()
  if execution_commit!=D2_RUNNER_EXACT or not checkout_clean: raise RuntimeError()
  runner_path=repo/RUNNER_PATH
  if not runner_path.is_file(): raise RuntimeError()
  subprocess.run([sys.executable,'-m','pip','install',str(repo)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
  source_direction_artifact_identity={'run_id':'geometry-v1-qk-direction-all-layer-41742d462d62','runner_execution_exact':'41742d462d62525189855c8ebb2ee1995fb9230a'}; runner_execution_identity={'commit':execution_commit,'clean':checkout_clean}
 except BaseException: fail('checkout')


In [ ]:
if not FAILED:
 rd=wr=None
 try:
  if ATTEMPTED: raise RuntimeError()
  ATTEMPTED=True; hf_token=userdata.get('HF_TOKEN')
  if not isinstance(hf_token,str) or not hf_token: raise RuntimeError()
  runner_env={k:v for k,v in os.environ.items() if 'TOKEN' not in k.upper()}; runner_env['HF_TOKEN']=hf_token
  OUTPUT_ROOT.mkdir(parents=True,exist_ok=True); run_dir=OUTPUT_ROOT/('Geometry-V1-QK-D2-'+execution_commit[:12]+'-'+time.strftime('%Y%m%dT%H%M%SZ',time.gmtime()))
  if run_dir.exists(): raise RuntimeError()
  rd,wr=os.pipe(); command=[sys.executable,str(runner_path),'--repo-root',str(repo),'--expected-exact',execution_commit,'--source-root',str(SOURCE_ROOT),'--output-root',str(run_dir),'--control-fd',str(wr)]
  process=subprocess.Popen(command,cwd=repo,env=runner_env,pass_fds=(wr,),stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL); os.close(wr);wr=None; rc=process.wait(timeout=7200); line=os.read(rd,MAX_CONTROL_BYTES+1)
  if len(line)>MAX_CONTROL_BYTES: raise RuntimeError()
  control=json.loads(line[len(SUCCESS_PREFIX):]) if line.startswith(SUCCESS_PREFIX.encode()) else (json.loads(line[len(FAILURE_PREFIX):]) if line.startswith(FAILURE_PREFIX.encode()) else {'status':'unavailable'})
  terminal={'source_direction_artifact_identity':source_direction_artifact_identity,'runner_execution_identity':runner_execution_identity,'status':control.get('d2_status',control.get('status')),'selected_layer_paths':control.get('fixed_layer_paths',[]),'science_denominator':0}
  print('CEGWM_GEOMETRY_V1_QK_D2_TERMINAL '+json.dumps(terminal,sort_keys=True,separators=(',',':')))
  if rc!=0 or control.get('status')!='success': raise RuntimeError()
 except BaseException: fail('runner')
 finally:
  hf_token=None; runner_env.pop('HF_TOKEN',None)
  if wr is not None: os.close(wr)
  if rd is not None: os.close(rd)
